# 03 Does the embedding model matter?

Self-contained: this notebook downloads NFCorpus and the models from the Hugging Face Hub, embeds everything, and computes every number and figure from scratch. It does not read `results/`, `cache/`, or any `silent_rag` module, so it runs on a fresh machine with only the pip packages below. Expect roughly 20-40 minutes on a laptop CPU: the corpus is embedded once per distinct model/sequence-length/prefix configuration (about nine), and `bge-base` is the slowest.

**Questions**
1. Are all-MiniLM, bge-small and e5-small really better than the baseline (multi-qa-MiniLM)?
2. Is the effect large or merely detectable?
3. Are results sensitive to prefixes, sequence length or a larger model?

**Approach**
Paired per-query differences on all labelled NFCorpus queries, with every model at **its own default `max_seq_length`** (the earlier forced-512 design is kept as a sensitivity row); cluster-bootstrap CIs (queries grouped by topic, because queries on one topic are not independent); sign-flip permutation tests with Holm correction over the three pre-specified contrasts; TOST equivalence; minimum detectable effect; sensitivity configurations. Each inferential section has a **"Why this test"** note giving the estimand, the unit of independence, the assumptions, the alternative that was rejected, and the Monte Carlo resolution.

**Scope.** NFCorpus only, the project's development set. SciFact and FiQA are held out for the final run, so this notebook deliberately does not load them.

In [ ]:
# Uncomment on a fresh environment.
# %pip install -q datasets sentence-transformers scikit-learn scipy pandas numpy matplotlib

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # macOS: OpenMP clash between libraries; harmless elsewhere
os.environ.setdefault("OMP_NUM_THREADS", "1")

import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from scipy import stats
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 30)

SEED = 0
np.random.seed(SEED)
K = 10
SPLITS = ["train", "validation", "test"]  # the Hub qrels name is "validation", not "dev"
GROUP_K = 50      # topic groups for cluster bootstrap / group sign-flip
B = 2000          # bootstrap / permutation draws
DELTA = 0.01      # equivalence margin in nDCG@10, fixed before looking at results
BASELINE = "multi-qa-MiniLM"
MAX_SEQ = None    # None = each model's own default max_seq_length (primary); forced lengths are sensitivity rows

NFCORPUS = dict(
    repo="BeIR/nfcorpus", revision="b5026a0e96e8a7ac4f95f482a596389289d46269",
    qrels="BeIR/nfcorpus-qrels", qrels_revision="a451b3b26d3ae1358f259c1a3a4dd61fcea35a65",
)
MODELS = {
    "multi-qa-MiniLM": dict(repo="sentence-transformers/multi-qa-MiniLM-L6-cos-v1", revision="b207367332321f8e44f96e224ef15bc607f4dbf0", query_prefix="", doc_prefix=""),
    "all-MiniLM": dict(repo="sentence-transformers/all-MiniLM-L6-v2", revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41", query_prefix="", doc_prefix=""),
    "bge-small": dict(repo="BAAI/bge-small-en-v1.5", revision="5c38ec7c405ec4b44b94cc5a9bb96e735b38267a", query_prefix="Represent this sentence for searching relevant passages: ", doc_prefix=""),
    "e5-small": dict(repo="intfloat/e5-small-v2", revision="ffb93f3bd4047442299a41ebb6fa998a38507c52", query_prefix="query: ", doc_prefix="passage: "),
}

## Helpers and data

Data loading, encoding, exact cosine search and nDCG@10 (graded gain, log2 discount, as in BEIR). Encoders and embeddings are kept in memory and reused across configurations that share them (for example `e5-small` with and without the query prefix shares document vectors with the no-prefix run).

**Tie-breaking differs from the benchmark runner.** This notebook breaks exact score ties by document index (stable sort). `silent_rag.pipeline.Retriever` uses FAISS `IndexFlatIP`, which breaks ties its own way. The corpus has 39 duplicate-document pairs and 29 queries tie at rank 10/11, so mean nDCG@10 here can differ from the runner's clean baseline in the fourth decimal. Contrasts between models are unaffected at the precision reported.

In [ ]:
def doc_text(title, text):
    return f"{(title or '').strip()} {(text or '').strip()}".strip()


def load_nfcorpus():
    c = NFCORPUS
    corpus = load_dataset(c["repo"], "corpus", split="corpus", revision=c["revision"])
    queries = load_dataset(c["repo"], "queries", split="queries", revision=c["revision"])
    qtext = {str(q["_id"]): q["text"] for q in queries}
    split_of, qrels = {}, {}
    for split in SPLITS:
        for row in load_dataset(c["qrels"], split=split, revision=c["qrels_revision"]):
            q = str(row["query-id"])
            if split_of.setdefault(q, split) == split:
                qrels.setdefault(q, {})[str(row["corpus-id"])] = int(row["score"])
    doc_ids = [str(d["_id"]) for d in corpus]
    docs = [doc_text(t, x) for t, x in zip(corpus["title"], corpus["text"])]
    return doc_ids, docs, {q: qtext[q] for q in qrels}, qrels, split_of


def ndcg_at_k(ranked, rels, k=K):
    dcg = sum(rels.get(d, 0) / np.log2(i + 2) for i, d in enumerate(ranked[:k]))
    ideal = sorted(rels.values(), reverse=True)[:k]
    idcg = sum(r / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


def search(D, Q, k=K):
    S = Q @ D.T
    # Stable full sort: exact score ties (the corpus contains duplicate documents) break by document index,
    # so the top-k does not depend on which search routine is used.
    return np.argsort(-S, axis=1, kind="stable")[:, :k]


doc_ids, docs, queries, qrels, split_of = load_nfcorpus()
qids = [q for q in queries if any(r > 0 for r in qrels[q].values())]   # queries with a relevant document
split = np.array([split_of[q] for q in qids])
rel = [{d for d, v in qrels[q].items() if v > 0} for q in qids]
print(len(docs), "docs;", len(qids), "queries;", pd.Series(split).value_counts().to_dict())

## Retrieval for every configuration

**Sequence length.** The primary comparison runs every model at **its own default `max_seq_length`**, read from the loaded model before anything is overridden. An earlier version forced every model to 512 tokens, which is not how any of them ships: `all-MiniLM` defaults to 256 and scored higher there. The other length (256 or 512) is a labelled sensitivity row for the two MiniLM models.

The four main models, then the sensitivity configurations: bge-small without its query instruction, e5-small without its prefixes (and with only the query prefix), the two MiniLM models at their non-default length, and `bge-base-en-v1.5`. All models, including `bge-base`, are pinned to a fixed Hub revision. (An earlier version looked up the newest `bge-base` revision at run time, so its row could change whenever the model repository was updated.)

In [ ]:
bge_base_rev = "a5beb1e3e68b9ab74eb54cfd186867f64f240e1a"   # resolved on 2026-10-15 and pinned, so the run is reproducible
MODELS["bge-base"] = dict(repo="BAAI/bge-base-en-v1.5", revision=bge_base_rev, query_prefix=MODELS["bge-small"]["query_prefix"], doc_prefix="")

_enc, _doc_emb, _q_emb = {}, {}, {}
DEFAULT_SEQ = {}   # each model's shipped max_seq_length, read before any override


def encoder(label, max_seq=None):
    """Return (encoder, resolved max_seq). max_seq=None means the model's own default."""
    spec = MODELS[label]
    if label not in DEFAULT_SEQ:
        e = SentenceTransformer(spec["repo"], revision=spec["revision"], device="cpu")
        DEFAULT_SEQ[label] = e.max_seq_length
        _enc[(label, e.max_seq_length)] = e
    max_seq = DEFAULT_SEQ[label] if max_seq is None else max_seq
    if (label, max_seq) not in _enc:
        e = SentenceTransformer(spec["repo"], revision=spec["revision"], device="cpu")
        e.max_seq_length = max_seq
        _enc[(label, max_seq)] = e
    return _enc[(label, max_seq)], max_seq


def encode(enc, texts, batch_size=32):
    return np.ascontiguousarray(enc.encode(list(texts), batch_size=batch_size, normalize_embeddings=True,
                                           convert_to_numpy=True, show_progress_bar=False), dtype=np.float32)


def run_config(label, max_seq=MAX_SEQ, query_prefix=None, doc_prefix=None):
    spec = MODELS[label]
    qp = spec["query_prefix"] if query_prefix is None else query_prefix
    dp = spec["doc_prefix"] if doc_prefix is None else doc_prefix
    enc, max_seq = encoder(label, max_seq)
    if (label, max_seq, dp) not in _doc_emb:
        _doc_emb[(label, max_seq, dp)] = encode(enc, [dp + t for t in docs])
    if (label, max_seq, qp) not in _q_emb:
        _q_emb[(label, max_seq, qp)] = encode(enc, [qp + queries[q] for q in qids])
    D, Q = _doc_emb[(label, max_seq, dp)], _q_emb[(label, max_seq, qp)]
    top = search(D, Q)
    ids = [[doc_ids[j] for j in row] for row in top]
    ndcg = np.array([ndcg_at_k(r, qrels[q]) for r, q in zip(ids, qids)])
    recall = np.array([len(r & set(i)) / len(r) for r, i in zip(rel, ids)])
    return {"Q": Q, "ids": ids, "ndcg": ndcg, "recall": recall, "max_seq": max_seq}


for label in MODELS:
    encoder(label)
print("default max_seq_length:", DEFAULT_SEQ)

# The non-default length for the two MiniLM models: 256 if the default is 512, else 512.
ALT = {m: (256 if DEFAULT_SEQ[m] == 512 else 512) for m in (BASELINE, "all-MiniLM")}
ALT_NAME = {m: f"{m} @{ALT[m]}" for m in ALT}

configs = {
    "multi-qa-MiniLM": dict(label="multi-qa-MiniLM"),
    "all-MiniLM": dict(label="all-MiniLM"),
    "bge-small": dict(label="bge-small"),
    "e5-small": dict(label="e5-small"),
    # sensitivity
    "bge-small, no query prefix": dict(label="bge-small", query_prefix=""),
    "e5-small, no prefixes": dict(label="e5-small", query_prefix="", doc_prefix=""),
    "e5-small, query prefix only": dict(label="e5-small", doc_prefix=""),
    **{ALT_NAME[m]: dict(label=m, max_seq=ALT[m]) for m in ALT},
    "bge-base": dict(label="bge-base"),
}
runs = {}
for cname, kw in configs.items():
    runs[cname] = run_config(**kw)
    print(cname, "| max_seq", runs[cname]["max_seq"], "| mean nDCG@10", round(float(runs[cname]["ndcg"].mean()), 4), flush=True)
nd = pd.DataFrame({k: r["ndcg"] for k, r in runs.items()})

## Accuracy on the test queries

The 323 test queries only, for a like-for-like view with the earlier analysis: nDCG@10 with a 95% bootstrap CI over queries, recall@10, and the paired difference to the baseline with a paired bootstrap CI. Queries are treated as independent here; the all-query analysis below accounts for topic dependence.

**Why this test**
- **Estimand:** mean nDCG@10 per model on the 323 test queries, and the mean paired difference from the baseline.
- **Unit of independence:** the query. This is knowingly optimistic, because test queries share topics and relevant documents. The view exists only for comparison with the earlier test-only tables, and the cluster analysis below is the primary one.
- **Assumptions:** the percentile bootstrap needs the sample mean to be roughly symmetric. With n = 323 and bounded nDCG it is (notebook 06 checks window means at n = 100).
- **Why not the obvious alternative:** a paired t-interval assumes the per-query differences are close to normal. They are zero-inflated and bounded, so the bootstrap is used, which relies only on the mean's own sampling distribution.
- **Resolution / Monte Carlo error:** B = 2,000 resamples. Monte Carlo error on a 2.5% percentile endpoint is about 0.001 nDCG here, well below the CI width.

In [ ]:
def boot_ci(x, n_boot=B, seed=SEED):
    x = np.asarray(x, float)
    rng = np.random.default_rng(seed)
    b = x[rng.integers(0, len(x), (n_boot, len(x)))].mean(axis=1)
    return float(x.mean()), *np.percentile(b, [2.5, 97.5])


mtest = split == "test"
main = ["multi-qa-MiniLM", "all-MiniLM", "bge-small", "e5-small"]
rows = []
for m in main:
    mean, lo, hi = boot_ci(nd[m][mtest])
    row = {"model": m, "n_queries": int(mtest.sum()), "ndcg@10": mean, "ci_lo": lo, "ci_hi": hi, "recall@10": runs[m]["recall"][mtest].mean()}
    if m != BASELINE:
        dm, dlo, dhi = boot_ci((nd[m] - nd[BASELINE])[mtest])
        row.update({"diff_vs_baseline": dm, "diff_ci_lo": dlo, "diff_ci_hi": dhi})
    rows.append(row)
pd.DataFrame(rows).round(4)

## Query groups

Primary grouping: KMeans (k = 50) on the baseline model's query embeddings (about 65 queries per group). All group-level bootstrap and sign-flip procedures below use these groups.

In [ ]:
groups = KMeans(GROUP_K, n_init=5, random_state=SEED).fit_predict(runs[BASELINE]["Q"])
uniq, inv = np.unique(groups, return_inverse=True)
members = [np.where(inv == g)[0] for g in range(len(uniq))]
print("groups:", len(uniq), "| sizes: min", min(map(len, members)), "median", int(np.median([len(m) for m in members])), "max", max(map(len, members)))

## Paired comparison with the baseline

All models are at their own default `max_seq_length`. For a contrast `d_i = nDCG_model(i) - nDCG_baseline(i)`:

- **CI (iid)** and **CI (cluster)**: percentile bootstrap over queries and over groups. Their width ratio is the design effect.
- **p (sign-flip, query)**: randomly flip the sign of each `d_i` (2,000 times) and compare the mean with the observed. Valid if each `d_i` is symmetric about zero under the null.
- **p (sign-flip, group)**: flip the sign of whole groups, which keeps within-group dependence. This is the primary p-value.
- **Holm** over the three pre-specified contrasts, applied to the group-level p-values.
- **dz** is the standardized paired effect (mean / SD of `d_i`).

**Why this test**
- **Estimand:** the mean paired difference in nDCG@10 between each model and the baseline, over all labelled NFCorpus queries.
- **Unit of independence:** the KMeans(50) topic group, not the query. Queries on one topic share relevant documents, so their differences are correlated. Resampling and sign-flipping whole groups keeps that correlation. Dependence extends beyond k = 50 (notebook 04), so the group CI is a lower bound on the uncertainty.
- **Assumptions:** the sign-flip test assumes each group's summed difference is symmetric about zero under "no difference". That holds if the two models are exchangeable within a group. The group bootstrap assumes groups are independent draws.
- **Why not the obvious alternative:** a paired t-test or Wilcoxon test over queries treats queries as independent, which the design effect (CI width ratio > 1) shows they are not. A t-test also assumes the differences are close to normal, and they are zero-inflated. Holm, not Bonferroni, because it is uniformly more powerful under the same error guarantee for three pre-specified contrasts.
- **Resolution / Monte Carlo error:** the sign-flip p-value has a floor of 1/(B+1) = 0.0005, so Holm over three contrasts has a floor of 0.0015. Values at the floor are reported as `p < 0.002`.

In [ ]:
def boot_mean(d, B=B, labels_members=None, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(d)
    if labels_members is None:
        return np.array([d[rng.integers(0, n, n)].mean() for _ in range(B)])
    out = np.empty(B)
    for b in range(B):
        idx = np.concatenate([labels_members[g] for g in rng.integers(0, len(labels_members), len(labels_members))])
        out[b] = d[idx].mean()
    return out


def signflip_p(d, B=B, members_=None, seed=SEED):
    rng = np.random.default_rng(seed)
    obs = d.mean()
    if members_ is None:
        null = np.array([(d * rng.choice([-1.0, 1.0], len(d))).mean() for _ in range(B)])
    else:
        sums = np.array([d[m].sum() for m in members_])
        null = np.array([(sums * rng.choice([-1.0, 1.0], len(sums))).sum() / len(d) for _ in range(B)])
    return (1 + (np.abs(null) >= abs(obs)).sum()) / (1 + B)


def holm(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    out, run = np.empty(len(p)), 0.0
    for rank, i in enumerate(order):
        run = max(run, min(1.0, p[i] * (len(p) - rank)))
        out[i] = run
    return out


def contrast(a, b=BASELINE, mask=None):
    d = (nd[a] - nd[b]).to_numpy()
    mem = members
    if mask is not None:
        d = d[mask]
        remap = {q: i for i, q in enumerate(np.where(mask)[0])}
        mem = [m for m in (np.array([remap[i] for i in m if i in remap]) for m in members) if len(m)]
    bi, bc = boot_mean(d), boot_mean(d, labels_members=mem)
    ci_i, ci_c = np.percentile(bi, [2.5, 97.5]), np.percentile(bc, [2.5, 97.5])
    return {"n": len(d), "mean_diff": d.mean(), "sd_d": d.std(ddof=1), "dz": d.mean() / d.std(ddof=1),
            "ci_lo_iid": ci_i[0], "ci_hi_iid": ci_i[1], "ci_lo_cluster": ci_c[0], "ci_hi_cluster": ci_c[1],
            "design_effect": np.ptp(ci_c) / np.ptp(ci_i), "p_signflip_query": signflip_p(d),
            "p_signflip_group": signflip_p(d, members_=mem), "d": d, "mem": mem}


primary = ["all-MiniLM", "bge-small", "e5-small"]
store, rows = {}, []
for m in primary:
    store[m] = contrast(m)
    rows.append({"model": m, **{k: v for k, v in store[m].items() if k not in ("d", "mem")}})
pair = pd.DataFrame(rows)
pair["p_holm_group"] = holm(pair["p_signflip_group"])
pair.insert(1, "mean_nDCG@10", [nd[m].mean() for m in primary])
print("baseline mean nDCG@10:", round(float(nd[BASELINE].mean()), 4), "| queries:", len(nd))
pair.round(4)

## Stability of the topic grouping

The topic groups come from KMeans, so they depend on a random seed. The primary analysis fixes seed 0, but the CIs and p-values should not hinge on it. This repeats the group-level CI and sign-flip p-value with five different KMeans seeds. The spread across seeds is the part of the uncertainty that comes from the arbitrary grouping, not from the data.

In [ ]:
rows = []
for gs in range(5):
    lab_s = KMeans(GROUP_K, n_init=5, random_state=gs).fit_predict(runs[BASELINE]["Q"])
    uniq_s, inv_s = np.unique(lab_s, return_inverse=True)
    members_s = [np.where(inv_s == g)[0] for g in range(len(uniq_s))]
    for m in primary:
        d = (nd[m] - nd[BASELINE]).to_numpy()
        lo, hi = np.percentile(boot_mean(d, labels_members=members_s), [2.5, 97.5])
        rows.append({"grouping_seed": gs, "model": m, "mean_diff": d.mean(), "ci_lo": lo, "ci_hi": hi, "p_signflip_group": signflip_p(d, members_=members_s)})
gs_table = pd.DataFrame(rows)
gs_table.groupby("model").agg(mean_diff=("mean_diff", "first"), ci_lo_min=("ci_lo", "min"), ci_lo_max=("ci_lo", "max"),
                              ci_hi_min=("ci_hi", "min"), ci_hi_max=("ci_hi", "max"), p_max=("p_signflip_group", "max")).round(4)

## Does the test-split result generalize?

If the model effects are the same across splits, a test-only comparison is consistent with the all-query one. Per-split mean difference with a cluster-bootstrap CI, and a permutation test of heterogeneity across splits. The statistic is the range of the per-split mean differences; the null shuffles split labels **within each KMeans group** (2,000 times).

**Why this test**
- **Estimand:** how much the model-vs-baseline difference varies across train, validation and test, measured by the range of the per-split means.
- **Unit of independence:** queries within a topic group are exchangeable with respect to the split label; whole topics are not. The earlier version shuffled split labels across all queries. That null mixes topics between splits, so if splits differ in topic mix it adds variation to the null that the observed statistic doesn't have, and it ignores the dependence that the rest of the notebook models. Permuting within groups keeps each group's split composition fixed, so the null varies only what the hypothesis is about.
- **Assumptions:** under "no heterogeneity", the split labels are exchangeable among queries of the same topic group.
- **Why not the obvious alternative:** Cochran's Q or a chi-square test on per-split estimates needs per-split standard errors that are independent across splits. Here the splits share topics and relevant documents, so they are not.
- **Resolution / Monte Carlo error:** 2,000 permutations, floor 1/2,001 ≈ 0.0005. The Monte Carlo SE of a p-value near 0.05 is about 0.005.

In [ ]:
def permute_within(labels_arr, members_, rng):
    out = labels_arr.copy()
    for m in members_:
        out[m] = rng.permutation(labels_arr[m])
    return out


rows = []
for m in primary:
    d = store[m]["d"]
    per = {}
    for s in SPLITS:
        c = contrast(m, mask=split == s)
        per[s] = c["mean_diff"]
        rows.append({"model": m, "split": s, "n": c["n"], "mean_diff": c["mean_diff"], "ci_lo_cluster": c["ci_lo_cluster"], "ci_hi_cluster": c["ci_hi_cluster"]})
    rng = np.random.default_rng(SEED)
    obs = np.ptp(list(per.values()))
    null = np.array([np.ptp([d[sh == s].mean() for s in SPLITS]) for sh in (permute_within(split, members, rng) for _ in range(2000))])
    rows.append({"model": m, "split": "heterogeneity (range of split means, within-group permutation)", "n": len(d), "mean_diff": obs,
                 "ci_lo_cluster": np.nan, "ci_hi_cluster": np.nan, "perm_p": (1 + (null >= obs).sum()) / 2001})
pd.DataFrame(rows).round(4)

## Equivalence (TOST)

A CI that includes zero is not evidence of no difference. The equivalence claim `|mean difference| < delta` is supported when the 90% cluster-bootstrap CI (equivalent to two one-sided 5% tests) lies entirely inside `(-delta, +delta)`. `delta = 0.01` nDCG@10 was fixed in advance, and a grid (0.005, 0.01, 0.02) shows how the conclusion depends on it. Applied to the three model contrasts, e5-small vs bge-small, and the default-versus-other-length contrast for both MiniLM models.

**Why this test**
- **Estimand:** whether the mean paired difference lies within ±delta, the size of difference treated as practically irrelevant.
- **Unit of independence:** the KMeans(50) group, as in the main comparison.
- **Assumptions:** the 90% percentile interval has approximately correct coverage. Each one-sided 5% test is then rejected exactly when the interval excludes the corresponding bound.
- **Why not the obvious alternative:** "the 95% CI includes zero" only fails to show a difference; it says nothing about equivalence. TOST is the standard way to test "no meaningful difference". It uses a 90% interval because each of its two one-sided tests is at 5%.
- **Resolution / Monte Carlo error:** B = 2,000. Monte Carlo error on a 5% endpoint is about 0.001, so a verdict whose endpoint sits within 0.001 of ±delta is fragile.

In [ ]:
def tost(a, b):
    d = (nd[a] - nd[b]).to_numpy()
    lo90, hi90 = np.percentile(boot_mean(d, labels_members=members), [5, 95])
    row = {"contrast": f"{a} vs {b}", "mean_diff": d.mean(), "ci90_lo": lo90, "ci90_hi": hi90}
    for delta in (0.005, 0.01, 0.02):
        row[f"equivalent_d{delta}"] = bool(lo90 > -delta and hi90 < delta)
    return row


pairs = [("all-MiniLM", BASELINE), ("bge-small", BASELINE), ("e5-small", BASELINE), ("e5-small", "bge-small"),
         (BASELINE, ALT_NAME[BASELINE]), ("all-MiniLM", ALT_NAME["all-MiniLM"])]
pd.DataFrame([tost(a, b) for a, b in pairs]).round(4)

## Minimum detectable effect

With `n` paired queries and SD of the paired difference `sd_d`, a two-sided 5% test with 80% power detects `|d| >= (z_0.975 + z_0.80) * sd_d / sqrt(n_eff)`. With group dependence the effective sample size is `n / design_effect^2`. The table uses the measured `sd_d` and design effect of each contrast, and gives the same quantity for a 323-query test-only design.

**Why this calculation**
- **Estimand:** the smallest true mean difference this design would detect with 80% power. It answers "is a test-only comparison large enough?", not "is there an effect?".
- **Unit of independence:** groups, folded in through `n_eff = n / design_effect^2`. Here the design effect is a ratio of CI widths, so its square is the variance inflation.
- **Assumptions:** a normal approximation for the mean difference, which is reasonable at n ≥ 323. The measured `sd_d` is treated as known.
- **Why not the obvious alternative:** simulated power would need a generative model for the per-query differences. The analytic form is transparent and only used to compare designs.
- **Resolution:** a planning number with no Monte Carlo error. It inherits the uncertainty in `sd_d` and in the design effect.

In [ ]:
z = stats.norm.ppf(0.975) + stats.norm.ppf(0.80)
rows = []
for m in primary:
    c = store[m]
    n_eff = c["n"] / c["design_effect"] ** 2
    rows.append({"model": m, "n": c["n"], "sd_d": c["sd_d"], "design_effect": c["design_effect"], "n_eff": n_eff,
                 "MDE_all_queries": z * c["sd_d"] / np.sqrt(n_eff), "MDE_323_queries_iid": z * c["sd_d"] / np.sqrt(323), "observed_diff": c["mean_diff"]})
pd.DataFrame(rows).round(4)

## Sensitivity: prefixes, sequence length and a larger model

Mean nDCG@10 for every configuration, with the paired difference to its natural reference and a cluster-bootstrap CI. These contrasts are **exploratory** (Benjamini-Hochberg across the rows). A missing or wrong prefix, or an unsuitable sequence length, would make a model look weaker for the wrong reason. The last rows repeat the three primary contrasts under the **old forced-512 design** (every model at 512 tokens), so the effect of the design change can be read off directly.

**Why this test**
- **Estimand:** for each configuration, the mean paired difference from its natural reference (the same model with its default settings, or bge-small for bge-base).
- **Unit of independence:** the KMeans(50) group, as in the main comparison.
- **Assumptions:** the same as for the group sign-flip and group bootstrap. BH assumes the p-values are independent or positively dependent, which is plausible here because the contrasts share the same queries.
- **Why not the obvious alternative:** Holm controls the family-wise error rate, which is too strict for exploratory screening. BH controls the false discovery rate, which matches the goal of flagging settings worth a closer look.
- **Resolution / Monte Carlo error:** the sign-flip floor is 0.0005.

In [ ]:
refs = {"bge-small, no query prefix": "bge-small", "e5-small, no prefixes": "e5-small", "e5-small, query prefix only": "e5-small",
        ALT_NAME[BASELINE]: BASELINE, ALT_NAME["all-MiniLM"]: "all-MiniLM", "bge-base": "bge-small"}


def at512(m):
    """Config name of model m at 512 tokens (its default run if the default is 512)."""
    if DEFAULT_SEQ[m] == 512:
        return m
    if m in ALT_NAME and ALT[m] == 512:
        return ALT_NAME[m]
    return None


# the old forced-512 design: each primary model at 512 vs the baseline at 512 (where both runs exist)
old_design = {f"{m} @512 vs {BASELINE} @512 (old forced-512 design)": (at512(m), at512(BASELINE))
              for m in primary if at512(m) and at512(BASELINE)}

rows = []
for cfg, ref in list(refs.items()) + list(old_design.values()):
    c = contrast(cfg, b=ref)
    rows.append({"configuration": cfg, "reference": ref, "mean_nDCG@10": nd[cfg].mean(), "mean_diff": c["mean_diff"],
                 "ci_lo_cluster": c["ci_lo_cluster"], "ci_hi_cluster": c["ci_hi_cluster"], "p_signflip_group": c["p_signflip_group"]})
for row, name in zip(rows[len(refs):], old_design):
    row["configuration"] = name
sens = pd.DataFrame(rows)
p = sens["p_signflip_group"].to_numpy()
order = np.argsort(p)
adj = np.minimum.accumulate((p[order] * len(p) / (np.arange(len(p)) + 1))[::-1])[::-1]
sens["p_bh"] = np.empty(len(p))
sens.loc[order, "p_bh"] = np.minimum(adj, 1.0)
sens.round(4)

In [ ]:
order_ = main + ["bge-base"]
fig, ax = plt.subplots(figsize=(6, 3))
for i, m in enumerate(order_):
    d = nd[m].to_numpy()
    b = boot_mean(d, labels_members=members, B=500)
    ax.errorbar(i, d.mean(), yerr=[[d.mean() - np.percentile(b, 2.5)], [np.percentile(b, 97.5) - d.mean()]], fmt="o", capsize=3)
ax.set_xticks(range(len(order_)), order_, rotation=20, fontsize=8)
ax.set_ylabel("mean nDCG@10 (95% cluster CI)")
fig.tight_layout()
plt.show()

## Agreement between models

Spearman correlation of per-query nDCG@10 and mean top-10 Jaccard overlap between each pair of the four main models, over all labelled queries. High nDCG correlation with low top-10 overlap would mean models succeed and fail on the same queries while retrieving different documents.

In [ ]:
def pair_matrix(fn):
    return pd.DataFrame([[fn(a, b) for b in main] for a in main], index=main, columns=main)


corr = pair_matrix(lambda a, b: stats.spearmanr(runs[a]["ndcg"], runs[b]["ndcg"])[0])
jac = pair_matrix(lambda a, b: np.mean([len(set(x) & set(y)) / len(set(x) | set(y)) for x, y in zip(runs[a]["ids"], runs[b]["ids"])]))
print("per-query nDCG Spearman"); display(corr.round(2))
print("top-10 Jaccard"); jac.round(2)

In [ ]:
import datasets, sentence_transformers, sklearn, scipy, torch
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
       "sklearn": sklearn.__version__, "torch": torch.__version__, "sentence_transformers": sentence_transformers.__version__,
       "datasets": datasets.__version__, "seed": SEED, "device": "cpu", "group_k": GROUP_K, "bootstrap_B": B,
       "equivalence_margin": DELTA, "bge_base_revision": bge_base_rev})

## Reproducibility check against the original analysis

The original analysis saved its results as tables under `results/tables/`. This notebook recomputes everything from scratch and was compared with those tables. Where a value differs, the cause was traced and, where it was a flaw in the method rather than a rounding effect, fixed here. **Two sources of tiny differences apply to every notebook:** (1) the corpus contains 39 pairs of identical document vectors, so 29 queries have an exact score tie at rank 10/11, and which document is kept depends on the search routine (mean nDCG@10 ranges 0.29581 to 0.29590 across routines). This notebook breaks ties by document index (stable sort), which makes the top-k deterministic. FAISS, which the benchmark runner uses, breaks them differently. (2) The package versions differ from those that wrote the saved tables, so differences in the third decimal are expected.

The original design forced every model to 512 tokens. That design is reproduced in the sensitivity table (the "old forced-512 design" rows), so the "Original" column can be compared like for like. The primary rows now use each model's default length (all-MiniLM 256; the others 512).

| Quantity | Original | This notebook | Status |
|---|---|---|---|
| Mean nDCG@10, all 3,237 queries (baseline / all-MiniLM / bge-small / e5-small), forced 512 | 0.2958 / 0.3124 / 0.3412 / 0.3252 | 0.2959 / 0.3126 / 0.3412 / 0.3253 | +/- 0.0002 (tie-breaking) |
| Paired difference vs baseline, forced 512 (all-MiniLM / bge-small / e5-small) | +0.0167 / +0.0454 / +0.0295 | +0.0167 / +0.0453 / +0.0294 | identical |
| **Paired difference vs baseline, default lengths (primary)** | not computed | **+0.0227** / +0.0453 / +0.0294 | **changed design** (all-MiniLM at its default 256) |
| Design effect (cluster CI width / iid width), default lengths | 1.46 / 1.37 / 1.26 | 1.46 / 1.35 / 1.24 | identical |
| Holm-adjusted p (three contrasts) | 0.0015 | 0.0015 | identical (resolution floor) |
| **By split: heterogeneity p (all-MiniLM / bge-small / e5-small)** | 0.459 / 0.494 / 0.054 (labels shuffled across all queries) | **0.449 / 0.433 / 0.039** (labels shuffled within topic groups) | **changed test** |
| Equivalence (delta = 0.01): which contrasts qualify | baseline 256 vs 512 and all-MiniLM 256 vs 512 | same (baseline also within 0.005; all-MiniLM within 0.01 but not 0.005) | identical |
| Sensitivity rows (bge no prefix, e5 no prefixes, e5 query prefix only, baseline 256, all-MiniLM length, bge-base) | -0.0042, -0.0047, -0.0012, +0.0002, +0.0060 (256 vs 512), +0.0239 | -0.0041, -0.0048, -0.0012, +0.0001, -0.0060 (512 vs default 256, so the sign flips), +0.0238 | identical |
| **nDCG@10 on the 323 test queries** (baseline / all-MiniLM / bge-small / e5-small) | 0.2967 / 0.3096 / 0.3430 / 0.3246 | 0.2973 / **0.3167** (at 256) / 0.3437 / 0.3255 | **tie-breaking +0.0006 to +0.0009; all-MiniLM now at its default length** |
| **Agreement between models: nDCG Spearman and top-10 Jaccard** (baseline vs all-MiniLM / bge-small / e5-small) | 0.906 / 0.855 / 0.907 and 0.301 / 0.259 / 0.271 | 0.88 / 0.86 / 0.88 and 0.32 / 0.28 / 0.29 | **changed** (all queries; all-MiniLM at 256) |
| **bge-base revision** | looked up at run time | pinned (a5beb1e3...) | fixed |

**Diagnosis.**
1. *Test-split nDCG, +0.0007 for every model (at matched lengths).* The shift is the same sign and size for all four models, which points to the 39 duplicate document pairs rather than to any model: when a judged-relevant document has an identical twin that is not judged, which of the two lands in the top 10 depends on how ties are broken. The saved tables came from FAISS tie-breaking; this notebook breaks ties by document index. On the 3,237-query set the effect averages out (0.0001 to 0.0002), and it is far below the width of the test-split CI (about +/- 0.035), so no conclusion changes.
2. *Agreement matrices.* The original computed these on the 323 test queries only (its failure tables also have 323 queries); this notebook uses all 3,237, with all-MiniLM at 256. Different query sets, so different values. The ordering is unchanged, with bge-small the most different from the baseline.
3. *bge-base was unpinned (an instability, fixed).* The original resolved the newest Hub revision at run time, so that row could change whenever the model repository was updated. The revision resolved on 2026-10-15 is now pinned.
4. *Dependence on the grouping seed.* Across five KMeans seeds the cluster CI endpoints move by at most about 0.002 (for example all-MiniLM: lower 0.0141 to 0.0156, upper 0.0298 to 0.0314) and the sign-flip p never exceeds 0.0005, so the conclusions do not depend on the seed.

**Second-review changes.**
- *Default sequence length (design change).* Forcing all-MiniLM to 512 tokens understated its gain over the baseline: +0.0167 at 512 against **+0.0227** at its default 256 (cluster CI 0.0154 to 0.0301). The three other models default to 512, so their rows are unchanged. The old design is kept as three sensitivity rows.
- *Heterogeneity test (method change).* Split labels are now permuted within KMeans groups. For e5-small the p-value moved from 0.055 to 0.039. That is below 0.05 unadjusted but not after Holm over the three models (3 × 0.039 = 0.117), so there is still no evidence that the effects differ by split. The e5-small validation difference (0.050) is the largest of its three splits.
- *Equivalence.* At delta = 0.01, all-MiniLM 256 vs 512 is equivalent (90% CI 0.0039 to 0.0082) even though it is also significantly different from zero (BH p = 0.0018). A difference can be real and still smaller than the margin.
- "Why this test" notes were added to every inferential section, and the FAISS tie-breaking caveat to the helpers.

## Findings
Numbers are from a full from-scratch run (seed 0, NFCorpus, 3,237 queries). Each model is at its own default `max_seq_length`: multi-qa-MiniLM 512, all-MiniLM 256, bge-small 512, e5-small 512, bge-base 512.

- **Three models beat the baseline** (mean nDCG@10 0.296): bge-small +0.045 (cluster CI 0.038 to 0.052), e5-small +0.029 (0.023 to 0.036), all-MiniLM +0.023 (0.015 to 0.030). All remain significant after Holm (adjusted p < 0.002, the resolution floor). Under the old forced-512 design all-MiniLM's gain was +0.017 (0.009 to 0.024). The other two rows do not change, because they already used their default length.
- **No evidence that the effects differ by split** (within-group permutation, heterogeneity p 0.45, 0.43, 0.039; after Holm over the three models the smallest is 0.117).
- **The effects are small.** Standardized paired effects are dz 0.16 to 0.29, which is 8% to 15% of the baseline's nDCG. No model pair is equivalent within 0.02, and none differs by as little as 0.01.
- **Test-only comparisons are underpowered.** On the 323 test queries the all-MiniLM interval now excludes zero but only just (+0.002 to +0.035). The minimum detectable effect is about 0.009 to 0.011 on all queries versus about 0.022 to 0.025 on 323. The test-only differences (0.019 to 0.046) are at or barely above that.
- **Prefixes matter slightly.** Dropping bge-small's query instruction costs 0.004 and dropping e5's prefixes costs 0.005 (both significant after BH, p = 0.010); keeping only e5's query prefix costs 0.001 (not significant).
- **Sequence length does not matter for the baseline** (256 vs its default 512: +0.0001, equivalent within 0.005) **but does for all-MiniLM**: 0.3186 at its default 256 against 0.3126 at 512 (-0.006 at 512, BH p = 0.002; equivalent within 0.01 but not 0.005).
- **A larger model helps more:** bge-base is +0.024 over bge-small (exploratory).
- **Models agree on which queries are hard but not on which documents to return:** per-query nDCG Spearman 0.86 to 0.91, mean top-10 Jaccard overlap 0.28 to 0.33.

### What this supports
On NFCorpus, using all 3,237 labelled queries with each model at its shipped settings, model choice among these small encoders changes mean nDCG@10 by a few hundredths, in a consistent order (bge-small > e5-small > all-MiniLM > baseline), with no evidence that the gaps differ across splits.

### What this does not support
- That the ranking, or the size of the gaps, holds on other datasets or domains (SciFact and FiQA are held out).
- That it generalizes to larger or differently trained encoders, or that it says why models differ.
- That the differences matter for downstream answer quality.
- That the p-values are exact: the cluster CIs use KMeans(50) groups, and dependence extends beyond them (see notebook 04), so the intervals are lower bounds on the uncertainty.
- That absolute nDCG measures quality: unjudged documents count as non-relevant, which penalizes models unlike the systems that built the judgment pool.
- That these numbers equal the benchmark runner's clean baseline to the fourth decimal: the runner breaks ties with FAISS, not a stable sort.
- That all-MiniLM's gain has one value: it is +0.017 to +0.023 depending on whether the length is forced to 512 or left at the default.